# Imports

In [ ]:
# only uncomment and install what is required
# !pip install imblearn
# !pip install scikit-learn
# !pip install imblearn
# !pip install xgboost

In [2]:
import pandas as pd
import numpy as np
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import traceback
from sklearn.metrics import f1_score

# Defining helper functions

In [15]:
# does a stratified train/test split
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold


def dataset_train_test_split_stratified_for_sub_type(df):
    try:
        df = df.dropna(how="any", axis=0)
        df.reset_index(drop=True, inplace=True)
        df.playlist_genre = pd.to_numeric(df.playlist_genre)

        if "data_type" not in df.columns:
            X_train, X_test, y_train, y_test = train_test_split(df.loc[:, df.columns != 'playlist_genre'],
                                                                df["playlist_genre"], test_size=0.30, random_state=0)
            df.loc[X_test.index, 'data_type'] = 'test'
            df.data_type = df.data_type.fillna('train')
        else:
            X_train = df[df['data_type'] == 'train']["text_clean"]
            y_train = df[df['data_type'] == 'train']["label"]

            X_test = df[df['data_type'] == 'test']["text_clean"]
            y_test = df[df['data_type'] == 'test']["label"]

        category, count = np.unique(y_train, return_counts=True)

        return X_train, y_train, X_test, y_test
    except Exception as e:
        print(e)

# generates a classification report
# labels = joblib.load(filename='misc/LabelDictionary.joblib')
def classification_report_generator(actual, predicted, model_name):
    report = classification_report(actual, predicted, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(
        f"{model_name}_classification_report.csv")
    return report

cv = StratifiedKFold(shuffle=True, random_state=0)

def weighted_f1_eval(y_pred, dtrain):
    y_true = dtrain.get_label()
    return 'weighted_f1_score', f1_score(y_true, y_pred, average='weighted'), True

# runs the grid search
def classifier_trainer(model, model_name, X_train, y_train, X_test, y_test, hyperparameters):
    try:
        if hyperparameters:
            # grid_search = GridSearchCV(model, hyperparameters, cv=cv, n_jobs=-1, scoring='f1_weighted')
            grid_search = RandomizedSearchCV(model, hyperparameters, cv=cv, n_jobs=-1, scoring='f1_weighted')
            grid_search.set_params(verbose = 10)
            grid_search.fit(X_train, y_train)
            model = grid_search.best_estimator_
            print(f"Best parameters for {model_name} found:", grid_search.best_params_)
        elif model_name =="XGB": 
          model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
        else:
          model.fit(X_train, y_train)  
        joblib.dump(model, f"{model_name}_sclf.sav")
        y_predicted = model.predict(X_test)
        report = classification_report_generator(y_test, y_predicted, model_name)

    except Exception as e:
        print("Error: %s, traceback: %s" % (e, traceback.format_exc()))

# Defining the Grid

In [38]:
parameter_tuning_dictionary = {"XGB": {
    'learning_rate': [0.01, 0.1, 0.5],
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 5]
    # 'gamma': [0.1, 0.2]
},
                               "RFC": {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
},
                               "ABC": {
    'n_estimators': [600, 700]
},
                               "GBC": {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 1.0],
    'max_depth': [3, 5, 7], 
},
                               "DT": {
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}}

# To do: Run the grid search

Uncomment the classifier you are planning to use and run the code

In [4]:
df = pd.read_csv("data/spotify_10000_dualgrams.csv", index_col=0)
df.drop(columns="lyrics", inplace=True)
X_train, y_train, X_test, y_test = dataset_train_test_split_stratified_for_sub_type(df)

Due to the time and resource limit, we ran the grid search parallelly.

Since we were not sure how long it would take for each grid search process, we put all codes in one file so that each person can select the codes they need and run it.

However, in the case of GBC and XGB, the running time exceeded even one day, even if we applied the early stop function.

So we had to give up grid search process for these 2 models and tune that models manually, but kept the codes we used here.

In [41]:
# DT = tree.DecisionTreeClassifier(random_state=0)
# classifier_trainer(DT, "DT", X_train, y_train, X_test, y_test, parameter_tuning_dictionary["DT"])

# RFC = RandomForestClassifier(random_state=0)
# classifier_trainer(RFC, "RFC", X_train, y_train, X_test, y_test, parameter_tuning_dictionary["RFC"])

# ABC = AdaBoostClassifier(n_estimators=400, random_state=0, learning_rate=0.1)
# classifier_trainer(ABC, "ABC", X_train, y_train, X_test, y_test, parameter_tuning_dictionary["ABC"])

# GBC = GradientBoostingClassifier(n_estimators=300, learning_rate=1.0, max_depth=5,
#                                           random_state=0)
# classifier_trainer(GBC, "GBC", X_train, y_train, X_test, y_test, parameter_tuning_dictionary["GBC"])


# in the case of XGB we tried to use the early stop function
from sklearn.metrics import f1_score

# Define the custom evaluation function to calculate f1-score
def f1_eval(y_pred, dtrain):
    # y_true = dtrain.get_label()
    # y_pred = y_pred.argmax(axis=1)
    return f1_score(dtrain, y_pred, average='weighted')

# early_stop = xgb.callback.EarlyStopping(
#     rounds=2, metric_name='mlogloss', save_best=True
# )
XGB = XGBClassifier(tree_method='hist', verbosity=2, n_jobs=None, eval_metric=f1_eval, n_estimators=150, max_depth=5, max_bin=10, objective='multi:softmax', random_state=0, gamma=0.001)
classifier_trainer(XGB, "XGB", X_train, y_train, X_test, y_test, hyperparameters=None)

[0]	validation_0-mlogloss:1.58101	validation_0-f1_eval:0.54550
[1]	validation_0-mlogloss:1.45923	validation_0-f1_eval:0.56207
[2]	validation_0-mlogloss:1.37509	validation_0-f1_eval:0.57545
[3]	validation_0-mlogloss:1.31641	validation_0-f1_eval:0.58407
[4]	validation_0-mlogloss:1.27366	validation_0-f1_eval:0.58208
[5]	validation_0-mlogloss:1.23717	validation_0-f1_eval:0.59027
[6]	validation_0-mlogloss:1.21044	validation_0-f1_eval:0.59633
[7]	validation_0-mlogloss:1.18738	validation_0-f1_eval:0.60001
[8]	validation_0-mlogloss:1.16918	validation_0-f1_eval:0.60016
[9]	validation_0-mlogloss:1.15571	validation_0-f1_eval:0.60432
[10]	validation_0-mlogloss:1.14459	validation_0-f1_eval:0.60525
[11]	validation_0-mlogloss:1.13314	validation_0-f1_eval:0.60742
[12]	validation_0-mlogloss:1.12466	validation_0-f1_eval:0.60988
[13]	validation_0-mlogloss:1.11625	validation_0-f1_eval:0.61099
[14]	validation_0-mlogloss:1.10853	validation_0-f1_eval:0.61288
[15]	validation_0-mlogloss:1.10199	validation_0-f1